# SDM invullen

### Imports

In [3]:
import sqlite3
import pandas as pd
import os

### Verbinding maken met SDM

In [4]:
db_path = sqlite3.connect("../../Database/SDM1.db")

# Cursor = object that allows interaction with the database by executing SQL commands and fetching query results
cursor = db_path.cursor()

### SDM schema dictionary

In [5]:
SDM_SCHEMA = {
    "Accessoire" :{
        "accessoirenr" : "INT",
        "naam" : "VARCHAR",
        "standaardprijs" : "float",
        "inkoopprijs": "float",
        "soort" : "VARCHAR",
        "leverancier": "INT"
    },
    "Accessoire_Inkoop":{
        "inkoopnr" : "INT",
        "inkoopmaand": "INT",
        "inkoopjaar": "INT",
        "aantal": "INT",
        "accessoire": "INT"
    }, 
    "Accessoireverkoop_Accessoire_Verkoop":{
        "äccessoire_verkoopnr" : "INT",
        "datun" : "VARCHAR",
        "aantal": "INT",
        "verkoopprijs" : "float",
        "Klant" : "INT",
        "accessoire" : "INT",
        "monteur" : "ÏNT"
    },
    "Fabrikant" :{
        "fabrikantnr" : "INT",
        "naam" : "VARCHAR",
        "adres" : "VARCHAR",
        "plaats" : "VARCHAR"
    },
    "Fiets" :{
        "fietsnr" : "INT",
        "soort" : "VARCHAR",
        "merk" : "VARCHAR",
        "type" : "VARCHAR",
        "standaartprijs" : "INT",
        "inkoopprijs" : "INT",
        "kleur" : "VARCHAR",
        "fabrikant" : "VARCHAR"
    },
    "Fiets_Inkoop":{
        "inkoopnr" : "INT",
        "inkoopmaand" : "INT",
        "inkoopjaar" : "INT",
        "aantal": "INT",
        "fiets" : "INT",
    },
    "Fietsverkoop_Fiets_Verkoop" : {
        "Fiets_verkoopnr" : "INT",
        "datum" : "INT",
        "aantal": "INT",
        "verkoopprijs" : "float",
        "klant" : "INT",
        "fiets" : "INT",
        "monteur" : "INT"
    },
    "Filiaal" : {
        "filialnr" : "INT",
        "naam" : "VARCHAR",
        "adres" : "VARCHAR",
        "provincie" : "VARCHAR"
    },
    "Klant" : {
        "klantnr" : "INT",
        "naam" : "VARCHAR",
        "adres" : "VARCHAR",
        "woonplaats" : "VARCHAR",
        "geslacht" : "VARCHAR",
        "geboortedatum" : "VARCHAR" 
    },
    "Leverancier" : {
        "leveranciernr" : "INT",
        "naam" : "VARCHAR",
        "adres" : "VARCHAR",
        "woonplaats" : "VARCHAR"
    },
    "Monteur" : {
        "monteurnr" : "INT",
        "naam" : "VARCHAR",
        "woonplaats" : "VARCHAR",
        "uurloon" : "FLOAT",
        "filiaal" : "INT"
    },
    "Onderhoud" : {
        "onderhoudnr" : "INT",
        "datum" : "DATE",
        "starttijd" : "TIME",
        "eindtijd" : "TIME",
        "fiets" : "INT",
        "monteur" : "INT"
    }
}

Dit zorgt er voor dat we later makelijker dingen kunnen doen zoals:
1. automatisch tabellen maken
2. automatisch tabellen leegmaken
3. automatisch data laden

### Reset_knop
a.	Alle tabellen uit het SDM leegmaken. Dit is dus een soort “reset-knop” voor de invulling van je SDM-tabellen.

In [ ]:
# Dit pakt alle tabelnamen uit onze schema
alle_tabellen = list(SDM_SCHEMA.keys())

# Je draait hier de volgorde om vanwegen foreign keys
alle_tabellen = alle_tabellen[::-1]

for tabel in alle_tabellen:
    delete_statement = f"DELETE FROM {tabel};"

    # Dit is de error handling
    try:
        cursor.execute(delete_statement)
        print(f"{tabel} geleegd")
    # Dit print de error voor debuggen
    except sqlite3.Error:
        print(f"FAILED: {delete_statement}")
        continue

db_path.commit()

### Bron databases openen

In [6]:
bron1 = sqlite3.connect("../../Database/BikeToDrive_1_Accessoireverkoop.db")
bron2 = sqlite3.connect("../../Database/BikeToDrive_2_Fietsverkoop.db")
bron3 = sqlite3.connect("../../Database/BikeToDrive_3_Onderhoud.db")
bron4 = sqlite3.connect("../../Database/BikeToDrive_4_Accessoire_Inkoop.db")
bron5 = sqlite3.connect("../../Database/BikeToDrive_5_Fiets_Inkoop.db")

### Data uit bron lezen

In [ ]:
# leverancier aleen uit "BikeToDrive_1_Accessoireverkoop", omdat bijde tabelen kwa inhoud presies het zelfte zijn.
df_leverancier = pd.read_sql_query(
    "SELECT * FROM Leverancier",
    bron1
)

# accessoire aleen uit "BikeToDrive_4_Accessoire_Inkoop", omdat die al de volledige dataset bevat.
df_accessoire = pd.read_sql_query(
    "SELECT * FROM Accessoire",
    bron4
)

df_accessoire_inkoop = pd.read_sql_query(
    "SELECT * FROM Accessoire_Inkoop",
    bron4
)

# filiaal aleen uit "BikeToDrive_3_Onderhoud", omdat die al de volledige dataset bevat
df_filiaal = pd.read_sql_query(
    "SELECT * FROM Filiaal",
    bron3
)

# Dezen moeten nog:
# Accessoireverkoop_Accessoire_Verkoop, Fabrikant, Fiets, Fiets_Inkoop, Fietsverkoop_Fiets_Verkoop, Klant, Monteur, Onderhoud

### Data overzetten naar SDM
b.	Data van elk .db-bestand overzetten naar het Source Data Model. Besteed hierbij extra aandacht aan de database-overschrijdende associaties.